# 06 — LiteLLM Router: Examples

> Use LiteLLM's built-in Router for multi-provider routing, fallback, and load balancing.

---

**What you'll build:**
1. Basic LiteLLM unified completion
2. Router with multiple deployments
3. Automatic fallback configuration
4. Multi-key load balancing
5. Cost tracking per call

In [ ]:
# !pip install litellm python-dotenv rich

In [ ]:
import os
import litellm
from litellm import Router, completion, completion_cost
from dotenv import load_dotenv
from rich import print as rprint
from rich.table import Table
from rich.console import Console

load_dotenv()
litellm.set_verbose = False   # Suppress debug output
console = Console()

OPENAI_KEY = os.getenv("OPENAI_API_KEY", "")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY", "")
GOOGLE_KEY = os.getenv("GOOGLE_API_KEY", "")

print('✅ Setup complete')

---
## Part 1: Unified Completion API

In [ ]:
import time

def litellm_call(model: str, prompt: str) -> dict:
    start = time.time()
    response = litellm.completion(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80
    )
    latency = (time.time() - start) * 1000
    cost = litellm.completion_cost(completion_response=response)
    return {
        "content": response.choices[0].message.content,
        "tokens": response.usage.total_tokens,
        "cost": cost,
        "latency_ms": round(latency, 1),
    }


prompt = "What is LLM routing? Answer in 1 sentence."

# ── Test across providers ──────────────────────────────────────────────────
models_to_test = [
    "gpt-4o-mini",                     # OpenAI
    # "claude-3-5-haiku-20241022",      # Anthropic (uncomment if you have key)
    # "gemini/gemini-1.5-flash",        # Google (uncomment if you have key)
]

table = Table(title="LiteLLM Unified API", show_header=True)
table.add_column("Model", style="cyan")
table.add_column("Latency", style="yellow")
table.add_column("Tokens", style="white")
table.add_column("Cost", style="green")
table.add_column("Response", style="white", max_width=50)

for model in models_to_test:
    try:
        result = litellm_call(model, prompt)
        table.add_row(
            model,
            f"{result['latency_ms']}ms",
            str(result["tokens"]),
            f"${result['cost']:.6f}",
            result["content"][:48] + ".." if len(result["content"]) > 48 else result["content"]
        )
    except Exception as e:
        table.add_row(model, "-", "-", "-", f"Error: {str(e)[:40]}")

console.print(table)

---
## Part 2: Router with Multiple Deployments

In [ ]:
# Configure a router with multiple model aliases
router = Router(
    model_list=[
        # 'fast' alias — two deployments for load balancing
        {
            "model_name": "fast",
            "litellm_params": {
                "model": "gpt-4o-mini",
                "api_key": OPENAI_KEY,
            }
        },
        # 'smart' alias — flagship model
        {
            "model_name": "smart",
            "litellm_params": {
                "model": "gpt-4o",
                "api_key": OPENAI_KEY,
            }
        },
    ],
    routing_strategy="simple-shuffle",
    num_retries=2,
    timeout=30,
)

print("✅ Router configured")
print(f"   Registered aliases: {list(set(m['model_name'] for m in router.model_list))}")

In [ ]:
# ── Call using alias ──────────────────────────────────────────────────────
prompts_and_models = [
    ("fast",  "Translate 'hello' to French."),
    ("fast",  "What is 15% of 200?"),
    ("smart", "Explain the concept of routing in distributed systems."),
]

table = Table(title="Router: Alias-Based Routing", show_header=True)
table.add_column("Alias", style="cyan")
table.add_column("Prompt", style="white", max_width=35)
table.add_column("Model Used", style="green")
table.add_column("Latency", style="yellow")
table.add_column("Cost", style="white")

for alias, prompt in prompts_and_models:
    start = time.time()
    response = router.completion(model=alias, messages=[{"role": "user", "content": prompt}], max_tokens=60)
    latency = (time.time() - start) * 1000
    cost = litellm.completion_cost(completion_response=response)
    table.add_row(
        alias,
        prompt[:33] + ".." if len(prompt) > 33 else prompt,
        response.model,
        f"{latency:.0f}ms",
        f"${cost:.6f}"
    )

console.print(table)

---
## Part 3: Fallback Configuration

In [ ]:
# Router with declarative fallbacks
router_with_fallback = Router(
    model_list=[
        {"model_name": "primary",  "litellm_params": {"model": "gpt-4o",      "api_key": OPENAI_KEY}},
        {"model_name": "backup",   "litellm_params": {"model": "gpt-4o-mini",  "api_key": OPENAI_KEY}},
    ],
    fallbacks=[
        {"primary": ["backup"]}   # If primary fails, try backup
    ],
    context_window_fallbacks=[
        {"primary": ["backup"]}   # If context limit exceeded, try backup
    ],
    num_retries=2,
    timeout=20,
)

# Normal request — goes to primary
response = router_with_fallback.completion(
    model="primary",
    messages=[{"role": "user", "content": "What is fallback routing?"}],
    max_tokens=60
)
rprint(f"\n✅ Response from: [green]{response.model}[/green]")
rprint(f"   {response.choices[0].message.content[:150]}")

---
## Part 4: Multi-Key Load Balancing

In [ ]:
# Multi-key balancing: same model alias, different API keys
# In production: OPENAI_KEY_1, OPENAI_KEY_2, etc.
# Here we use the same key twice for demonstration

multi_key_router = Router(
    model_list=[
        {
            "model_name": "gpt4mini_pool",
            "litellm_params": {
                "model": "gpt-4o-mini",
                "api_key": OPENAI_KEY,
            },
            # In production, set TPM/RPM to help router respect limits:
            # "tpm": 200_000,
            # "rpm": 3_500,
        },
        {
            "model_name": "gpt4mini_pool",
            "litellm_params": {
                "model": "gpt-4o-mini",
                "api_key": OPENAI_KEY,   # Different key in production
            },
        },
    ],
    routing_strategy="least-busy",
)

# Send several requests
questions = [
    "What is load balancing?",
    "What is horizontal scaling?",
    "What is a rate limit?",
]

print("📤 Running load-balanced requests through multi-key pool...\n")
total_cost = 0
for q in questions:
    resp = multi_key_router.completion(
        model="gpt4mini_pool",
        messages=[{"role": "user", "content": q}],
        max_tokens=50
    )
    cost = litellm.completion_cost(completion_response=resp)
    total_cost += cost
    rprint(f"   [cyan]{q}[/cyan]")
    rprint(f"   → model={resp.model}, cost=${cost:.6f}")
    rprint(f"   {resp.choices[0].message.content[:80]}\n")

rprint(f"[bold]Total cost for {len(questions)} requests: ${total_cost:.5f}[/bold]")

---
## Part 5: Async Completion for High Throughput

In [ ]:
import asyncio

async def async_batch(router, prompts: list[str]) -> list[str]:
    """Run multiple prompts concurrently through the router."""
    tasks = [
        router.acompletion(
            model="gpt4mini_pool",
            messages=[{"role": "user", "content": p}],
            max_tokens=50
        )
        for p in prompts
    ]
    responses = await asyncio.gather(*tasks)
    return [r.choices[0].message.content for r in responses]


batch_prompts = [
    "Define routing in one sentence.",
    "Define fallback in one sentence.",
    "Define load balancing in one sentence.",
    "Define circuit breaker in one sentence.",
]

print(f"🚀 Running {len(batch_prompts)} requests concurrently...\n")
start = time.time()
results = await async_batch(multi_key_router, batch_prompts)
total_time = (time.time() - start) * 1000

for prompt, result in zip(batch_prompts, results):
    print(f"Q: {prompt}")
    print(f"A: {result[:100]}\n")

print(f"⏱️  All {len(batch_prompts)} requests completed in {total_time:.0f}ms (concurrent)")

---
## Summary

| Feature | How to use |
|---|---|
| Unified completion | `litellm.completion(model=..., messages=...)` |
| Router with aliases | `Router(model_list=[...])` + `router.completion(model=alias)` |
| Fallback | `Router(fallbacks=[{"primary": ["backup"]}])` |
| Multi-key balancing | Multiple entries with same `model_name`, different `api_key` |
| Cost tracking | `litellm.completion_cost(response)` |
| Async / high throughput | `router.acompletion()` + `asyncio.gather()` |

**Next:** `07_production_router` — combine everything into a single production-ready `ProductionRouter` class.